# 🛡️ Agent Engineering Challenge

**Duration:** ~60 minutes &nbsp;|&nbsp; **Solo build (peer collaboration encouraged)** &nbsp;|&nbsp; **Claude Messages API**

---

## What You're Building

An AI agent that automates RFP (Request for Proposal) responses for a cybersecurity vendor. Your agent will:

1. **Parse** a questionnaire into categorized questions
2. **Retrieve** relevant source material from a knowledge base
3. **Draft** polished, cited answers
4. **Review** answers for cross-question consistency
5. **Export** structured JSON output

The questionnaire in this notebook is what you'll develop and test against. At the end of the session you'll get a **surprise RFP** to point your agent at — so the goal isn't to hardcode the questions below, it's to build something flexible and robust enough to handle questions you haven't seen yet.

This is a solo build, but you're strongly encouraged to bounce ideas off the people around you — sketch architectures together, compare tool designs, debug each other's tool-use loops. Each person ships their own agent.

### Design Principles to Keep in Mind

- **Generality over specificity.** Tool design, prompts, and parsing should not assume the exact questions below.
- **Failure modes matter.** What happens when the KB doesn't have the answer? When a question spans two categories? When the prospect asks something off-script?
- **Consistency is the real quality bar.** Cross-question contradictions are the #1 failure mode of automated RFP responses — your review step is what separates a toy from something deployable.

### Build Something Worth Showing Off

There's no formal demo and no grading. But there's a room full of smart people who'd love to see something interesting. Past Partner Basecamp builds we've loved: RFP deck builders, robust eval harnesses, analysis UIs to visualize answer accuracy, search UIs that plug into any RFP, agents that route by category to specialized subagents. If you ship the baseline early, spend the remaining time building the thing **you** want to show off.

### Suggested Time Allocation

| Phase | Duration | Focus |
|-------|----------|-------|
| Setup & Planning | 5 min | Verify API, sketch architecture, identify your robustness bets |
| Build | 45 min | Agent loop, prompt design, testing on the included questionnaire |
| Surprise RFP | 5–8 min | Point your agent at the unseen RFP — see what holds up |

Extra time? Push harder on robustness, add tooling, or build the show-off feature you've been sitting on.

---

## Part 0: Environment Setup

Run the cells below to install dependencies and verify your API connection.

In [ ]:
# Install dependencies
!pip install anthropic --upgrade -q

print("✓ Dependencies installed")

In [ ]:
import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional

import anthropic

# === API KEY SETUP ===
ANTHROPIC_API_KEY = ""  # <-- Paste your Anthropic API key here

if not ANTHROPIC_API_KEY:
    raise ValueError(
        "❌ ANTHROPIC_API_KEY not found.\n"
        "Set it via: export ANTHROPIC_API_KEY='your-key-here'\n"
        "Or paste directly in the cell above."
    )

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("✓ Anthropic client initialized")

In [ ]:
# Verify API connectivity
try:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=20,
        messages=[{"role": "user", "content": "Say 'ready' and nothing else."}]
    )
    print(f"✓ API connection verified: {response.content[0].text}")
    print(f"  Model: {response.model}")
except Exception as e:
    print(f"❌ API connection failed: {e}")

---

## Part 1: The Brief

### The Situation

**Helios Security** is a mid-market cybersecurity vendor selling endpoint protection, SIEM, and managed detection & response. Their sales team responds to **40+ RFPs per quarter**, each containing 50–200 questions spanning technical architecture, compliance certifications, pricing models, and company background.

Today, each RFP takes a solutions engineer **6–8 hours** to complete. The process is manual: hunt through a Confluence wiki of past proposals, cross-reference product documentation, and copy-paste answers into a spreadsheet — adjusting tone and specificity for each prospect. Answers frequently **contradict each other** across questions because no one reviews the full response holistically.

Helios wants an AI agent that takes in an RFP questionnaire and produces a draft response in minutes — decomposing questions, retrieving relevant source material, synthesizing polished answers, and flagging anything the human needs to review.

**The goal:** Cut first-draft time from 8 hours to under 15 minutes and eliminate cross-answer inconsistencies.

### The Agent Pipeline

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  PARSE   │───▶│ RETRIEVE │───▶│  DRAFT   │───▶│  REVIEW  │───▶│  EXPORT  │
│          │    │          │    │          │    │          │    │          │
│ Break    │    │ Search   │    │ Generate │    │ Check    │    │ Return   │
│ into Qs  │    │ mock KB  │    │ answers  │    │ for      │    │ struct.  │
│ + tag    │    │ via tool │    │ w/ cites │    │ contra-  │    │ JSON     │
│ category │    │          │    │          │    │ dictions │    │          │
└──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
```

### Illustrative RFP Questions

The kinds of questions Helios sees in real RFPs span multiple categories — some have crisp answers in the KB, some don't, and some span two categories at once. Examples of the shape (these are **not** the exact questions you'll build against — those are in Part 5, and the surprise RFP at the end will be different again):

| ID | Category | Question |
|----|----------|----------|
| Ex1 | Technical | What is your incident response SLA for a confirmed P1 event? How is escalation handled outside business hours? |
| Ex2 | Compliance | Describe your sub-processor management process and how you notify customers of changes. |
| Ex3 | Pricing | What pricing flexibility is available for multi-year commitments, and how are price increases capped at renewal? |
| Ex4 | Company-Info | What is your customer support model? Describe tiers, channels, and average response times. |
| Ex5 | Technical + Compliance | How does your platform support customer-controlled key management, and which regions are FIPS 140-2 validated? |

Some of these are well-covered by the knowledge base. Some aren't. Your agent should handle both gracefully.

---

## Part 2: Mock Knowledge Base (Pre-Built)

Your agent needs data to retrieve from. Below is a pre-built mock knowledge base with realistic Helios Security content. **You can extend this with additional entries**, but the baseline is ready to use.

In [ ]:
# ============================================================
# MOCK KNOWLEDGE BASE
# Pre-populated with realistic Helios Security content.
# Extend with additional entries if you want richer agent behavior.
# ============================================================

KNOWLEDGE_BASE = {
    "threat_detection": {
        "source": "Helios Platform Architecture Doc v4.2",
        "content": (
            "Helios Sentinel uses a multi-layered detection engine combining "
            "signature-based matching, behavioral analysis, and ML-driven anomaly detection. "
            "Data sources include endpoint telemetry (process events, file system changes, "
            "network connections), cloud workload logs (AWS CloudTrail, Azure Activity Log, "
            "GCP Audit Log), network flow data (NetFlow v9/IPFIX), and email gateway events. "
            "Average detection-to-alert latency is 2.3 seconds for signature matches and "
            "18 seconds for behavioral detections. Our SIEM correlation engine processes "
            "up to 50,000 events per second per tenant."
        ),
        "tags": ["technical", "detection", "latency", "architecture"]
    },
    "compliance_certs": {
        "source": "Helios Compliance & Certifications Register 2025",
        "content": (
            "Current certifications: SOC 2 Type II (audited December 2024 by Deloitte), "
            "ISO 27001:2022 (certified March 2024 by BSI), FedRAMP Moderate (authorized "
            "June 2024, sponsored by DHS), HIPAA (BAA available, last assessment October 2024), "
            "PCI DSS v4.0 Level 1 Service Provider (validated September 2024 by Coalfire). "
            "StateRAMP authorized (January 2025). All certifications maintained on continuous "
            "monitoring basis with quarterly internal audits."
        ),
        "tags": ["compliance", "certifications", "audit", "soc2", "fedramp"]
    },
    "pricing_model": {
        "source": "Helios Commercial Pricing Sheet Q1 2025",
        "content": (
            "Endpoint Protection Platform (EPP+EDR bundle): "
            "500 endpoints: $18/seat/month ($108,000/year). "
            "1,000 endpoints: $15/seat/month ($180,000/year) — 17% volume discount. "
            "5,000 endpoints: $11/seat/month ($660,000/year) — 39% volume discount. "
            "Minimum contract term: 12 months. Multi-year discounts: 2-year = additional 5%, "
            "3-year = additional 10%. SIEM add-on: +$6/seat/month. "
            "MDR add-on: +$12/seat/month. All pricing excludes professional services."
        ),
        "tags": ["pricing", "commercial", "discount", "contract"]
    },
    "financial_services_customers": {
        "source": "Helios Customer Success — Vertical Report 2024",
        "content": (
            "Helios currently serves 47 customers in financial services, including "
            "12 banks, 8 insurance carriers, 15 asset management firms, and 12 fintech companies. "
            "Reference accounts (approved for external use): "
            "1) Meridian National Bank — 3,200 endpoints, EPP+EDR+SIEM, deployed since 2022. "
            "2) Crestview Capital Partners — 850 endpoints, EPP+MDR, deployed since 2023. "
            "3) Apex Insurance Group — 5,100 endpoints, full platform, deployed since 2021. "
            "Average NPS in financial services vertical: 72."
        ),
        "tags": ["company-info", "customers", "financial-services", "references"]
    },
    "data_residency_eu": {
        "source": "Helios Data Sovereignty & Privacy Whitepaper v3.1",
        "content": (
            "Helios supports full EU data residency through dedicated infrastructure in "
            "Frankfurt (AWS eu-central-1) and Dublin (AWS eu-west-1). Customer data never "
            "leaves the selected region. Encryption at rest: AES-256-GCM with customer-managed "
            "keys (AWS KMS or BYOK). Encryption in transit: TLS 1.3 for all API and agent "
            "communications, with certificate pinning for endpoint agents. "
            "GDPR Data Processing Agreement (DPA) included in all EU contracts. "
            "Annual third-party penetration testing by NCC Group. "
            "Data retention: configurable per tenant, default 90 days for raw telemetry, "
            "13 months for aggregated alerts."
        ),
        "tags": ["technical", "compliance", "data-residency", "eu", "encryption", "gdpr"]
    },
    "past_rfp_detection_answer": {
        "source": "Acme Corp RFP Response — March 2024",
        "content": (
            "Q: Describe your real-time threat detection capabilities. "
            "A: Helios Sentinel provides sub-3-second detection for known threat patterns "
            "and under 20 seconds for behavioral anomalies. Our detection engine ingests "
            "endpoint telemetry, network flows, cloud audit logs, and email events. "
            "The SIEM correlation engine handles 50K EPS per tenant. "
            "We maintain a 99.7% true positive rate on our top 100 detection rules, "
            "validated quarterly against MITRE ATT&CK framework."
        ),
        "tags": ["technical", "detection", "past-rfp"]
    },
    "past_rfp_compliance_answer": {
        "source": "NovaTech RFP Response — July 2024",
        "content": (
            "Q: What compliance certifications do you hold? "
            "A: Helios holds SOC 2 Type II, ISO 27001, FedRAMP Moderate, PCI DSS v4.0, "
            "and HIPAA compliance. All certifications are actively maintained with "
            "continuous monitoring. We provide audit reports upon request under NDA. "
            "Our security team of 14 full-time engineers manages compliance programs."
        ),
        "tags": ["compliance", "certifications", "past-rfp"]
    }
}


def search_knowledge_base(query: str, category: Optional[str] = None) -> list[dict]:
    """
    Search the mock knowledge base.
    Returns matching entries ranked by keyword overlap.
    """
    query_terms = set(query.lower().split())
    results = []

    for entry_id, entry in KNOWLEDGE_BASE.items():
        # Score by keyword overlap with content + tags
        entry_text = (entry["content"] + " " + " ".join(entry["tags"])).lower()
        overlap = len(query_terms & set(entry_text.split()))

        # Boost if category matches a tag
        if category and category.lower() in [t.lower() for t in entry["tags"]]:
            overlap += 5

        if overlap > 0:
            results.append({
                "id": entry_id,
                "source": entry["source"],
                "content": entry["content"],
                "relevance_score": overlap
            })

    # Sort by relevance, return top 3
    results.sort(key=lambda x: x["relevance_score"], reverse=True)
    return results[:3]


# Quick test
test_results = search_knowledge_base("threat detection latency", category="technical")
print(f"✓ Knowledge base loaded ({len(KNOWLEDGE_BASE)} entries)")
print(f"  Test search 'threat detection latency': {len(test_results)} results")
print(f"  Top result: {test_results[0]['source']}")

---

## Part 3: Tool Definition

Define the `search_kb` tool so Claude can retrieve information from the knowledge base during its agent loop.

In [ ]:
# Tool definition for the Claude API
SEARCH_KB_TOOL = {
    "name": "search_kb",
    "description": (
        "Search the Helios Security knowledge base for information relevant to "
        "answering an RFP question. Returns up to 3 matching documents with source "
        "attribution. Use this to find product docs, past proposal answers, compliance "
        "records, and pricing information."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query — use keywords from the RFP question"
            },
            "category": {
                "type": "string",
                "enum": ["technical", "compliance", "pricing", "company-info"],
                "description": "Optional category filter to narrow results"
            }
        },
        "required": ["query"]
    }
}


def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute a tool call and return the result as a string."""
    if tool_name == "search_kb":
        results = search_knowledge_base(
            query=tool_input["query"],
            category=tool_input.get("category")
        )
        return json.dumps(results, indent=2)
    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})


print("✓ Tool definition ready")

---

## Part 4: Level 0 Agent (Working Baseline)

Below is a working agent that handles a **single question** end-to-end. It:
- Sends the question to Claude with the `search_kb` tool available
- Handles the tool use loop (Claude calls the tool → we execute it → send results back)
- Returns a structured answer

**Run this to confirm the agent loop works**, then extend it in Part 5.

In [ ]:
SYSTEM_PROMPT = """You are an AI assistant helping Helios Security respond to RFP questionnaires.

For each question, you must:
1. Use the search_kb tool to find relevant source material
2. Draft a professional, detailed answer grounded in the retrieved sources
3. Cite your sources by name
4. If the knowledge base doesn't contain enough information, flag the answer as low-confidence

Return your answer as JSON with this structure:
{
    "question_id": "Q1",
    "category": "technical",
    "answer": "Your drafted answer here...",
    "sources": ["Source Name 1", "Source Name 2"],
    "confidence": "high" | "medium" | "low",
    "flags": ["any concerns or notes for human review"]
}

Be specific, professional, and concise. Use concrete numbers from the source material."""


def answer_single_question(
    question_id: str,
    question_text: str,
    category: str,
    model: str = "claude-sonnet-4-20250514",
) -> dict:
    """
    Level 0 agent: answers a single RFP question with tool use.
    Handles the full tool use loop.
    """
    messages = [
        {
            "role": "user",
            "content": (
                f"Answer this RFP question.\n\n"
                f"Question ID: {question_id}\n"
                f"Category: {category}\n"
                f"Question: {question_text}\n\n"
                f"Search the knowledge base for relevant information, then draft your answer."
            )
        }
    ]

    # Agent loop: keep going until Claude stops calling tools
    max_turns = 5  # Safety limit
    for turn in range(max_turns):
        response = client.messages.create(
            model=model,
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=messages,
            tools=[SEARCH_KB_TOOL],
        )

        # If Claude is done (no more tool calls), extract the answer
        if response.stop_reason == "end_turn":
            # Find the text block with the JSON answer
            for block in response.content:
                if block.type == "text":
                    try:
                        # Try to parse JSON from the response
                        text = block.text
                        # Handle markdown code blocks
                        if "```json" in text:
                            text = text.split("```json")[1].split("```")[0]
                        elif "```" in text:
                            text = text.split("```")[1].split("```")[0]
                        return json.loads(text.strip())
                    except json.JSONDecodeError:
                        return {"raw_response": block.text, "parse_error": True}
            break

        # If Claude wants to use a tool, execute it and continue
        if response.stop_reason == "tool_use":
            # Add assistant's response to message history
            messages.append({"role": "assistant", "content": response.content})

            # Execute each tool call and collect results
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = handle_tool_call(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            messages.append({"role": "user", "content": tool_results})

    return {"error": "Max turns reached without completing"}


print("✓ Level 0 agent defined")

In [ ]:
# Test: Answer Q1
print("Answering Q1 (threat detection)...")
result = answer_single_question(
    question_id="Q1",
    question_text=(
        "Describe your platform's approach to real-time threat detection. "
        "What data sources are ingested, and what is the average detection-to-alert latency?"
    ),
    category="technical",
)

print(json.dumps(result, indent=2))

---

## Part 5: 🛠️ YOUR TASK — Multi-Question Agent

The Level 0 agent handles one question at a time. Your job is to build a **full pipeline** that:

1. **Accepts a list of RFP questions** (the full questionnaire)
2. **Processes each question** through the agent loop
3. **Collects all answers** into a structured response
4. **Exports as JSON** with all required fields

### Approaches to Consider

- **Sequential:** Process questions one by one using `answer_single_question()`. Simple, reliable, but slower.
- **Batch prompt:** Send all questions to Claude in a single prompt and let it call the tool multiple times. Faster, but harder to control.
- **Hybrid:** Parse and categorize first, then process by category to reuse retrieved context.

### Starter Questions

In [ ]:
# RFP Questionnaire
RFP_QUESTIONS = [
    {
        "id": "Q1",
        "category": "technical",
        "text": (
            "Describe your platform's approach to real-time threat detection. "
            "What data sources are ingested, and what is the average detection-to-alert latency?"
        ),
    },
    {
        "id": "Q2",
        "category": "compliance",
        "text": (
            "List all compliance certifications your organization currently holds "
            "(SOC 2, ISO 27001, FedRAMP, etc.) and the date of most recent audit for each."
        ),
    },
    {
        "id": "Q3",
        "category": "pricing",
        "text": (
            "Provide per-seat pricing for 500, 1,000, and 5,000 endpoints. "
            "Are volume discounts available? Is there a minimum contract term?"
        ),
    },
    {
        "id": "Q4",
        "category": "company-info",
        "text": (
            "How many customers do you currently serve in the financial services vertical? "
            "Provide 2–3 reference accounts."
        ),
    },
    {
        "id": "Q5",
        "category": "technical",
        "text": (
            "How does your platform handle data residency requirements for customers "
            "operating in the EU? Describe encryption at rest and in transit."
        ),
    },
]

In [ ]:
def process_rfp(questions: list[dict]) -> list[dict]:
    """Process a full RFP questionnaire and return structured answers."""
    answers = [None] * len(questions)

    # Fan out one thread per question so all agent loops run concurrently.
    # future_to_idx preserves original question order in the output list
    # regardless of which thread finishes first.
    with ThreadPoolExecutor(max_workers=len(questions)) as executor:
        future_to_idx = {
            executor.submit(
                answer_single_question,
                q["id"],
                q["text"],
                q["category"],
            ): i
            for i, q in enumerate(questions)
        }

        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                answers[idx] = future.result()
            except Exception as e:
                # One failed question should not abort the whole batch
                answers[idx] = {
                    "question_id": questions[idx]["id"],
                    "error": str(e),
                    "confidence": "low",
                    "flags": ["Failed to generate answer"],
                }

    return answers


# Run it
print("Processing RFP...")
all_answers = process_rfp(RFP_QUESTIONS)

# Display results
if all_answers:
    for ans in all_answers:
        q_id = ans.get('question_id', '?')
        conf = ans.get('confidence', '?')
        print(f"  {q_id}: confidence={conf}, sources={len(ans.get('sources', []))}")
    print(f"\n✓ Processed {len(all_answers)} questions")
else:
    print("❌ No answers returned — implement process_rfp() above")

---

## Part 6: 🛠️ YOUR TASK — Consistency Review Step

The customer specifically asked for **cross-answer consistency checking**. Answers drafted individually often contradict each other (e.g., Q2 says "FedRAMP authorized June 2024" but Q5 says "FedRAMP certified in 2023").

Build a review step that:
1. Takes all drafted answers as input
2. Identifies potential contradictions or inconsistencies
3. Flags issues for human review

**This is what separates a good agent from a great one.** In real deployments, consistency is the #1 quality issue in automated RFP responses.

In [ ]:
def review_answers(answers: list[dict]) -> dict:
    """
    Review all drafted answers for cross-question consistency.
    Returns a review report with any flagged issues.
    """
    # Format all answers into a single readable block for Claude
    answers_text = "\n\n".join(
        f"[{ans.get('question_id', '?')} — {ans.get('category', '?')}]\n"
        f"Answer: {ans.get('answer', '')}\n"
        f"Sources: {', '.join(ans.get('sources', []))}\n"
        f"Confidence: {ans.get('confidence', '?')}"
        for ans in answers
    )

    review_prompt = f"""You are reviewing a set of RFP answers drafted by an AI agent for Helios Security.
Your job is to identify cross-answer inconsistencies that a human reviewer should fix before submission.

Check for:
- Contradictory facts (e.g., conflicting certification dates, pricing figures, or customer counts)
- Tone mismatches (e.g., one answer is vague where another is specific on the same topic)
- Missing information that is implied by another answer but not stated
- Overconfidence flags (answers marked high-confidence but citing sparse sources)

Answers to review:
{answers_text}

Return a JSON review report with this structure:
{{
    "overall_quality": "high" | "medium" | "low",
    "issues": [
        {{
            "type": "contradiction" | "tone_mismatch" | "missing_info" | "confidence_mismatch",
            "questions_affected": ["Q1", "Q2"],
            "description": "Specific description of the issue",
            "recommendation": "What the human reviewer should do"
        }}
    ],
    "summary": "One-paragraph overall assessment"
}}

If there are no issues, return an empty issues list with overall_quality "high"."""

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=2048,
        messages=[{"role": "user", "content": review_prompt}],
    )

    text = response.content[0].text
    # Strip markdown code fences if present
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0]
    elif "```" in text:
        text = text.split("```")[1].split("```")[0]

    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        return {"raw_response": text, "parse_error": True}


# Run review on your answers
if all_answers:
    print("Running consistency review...")
    review = review_answers(all_answers)
    print(json.dumps(review, indent=2))
else:
    print("⚠️ No answers to review yet — complete Part 5 first")

---

## Part 7: Export Final Output

Package your agent's output as clean JSON — the deliverable Helios would actually receive.

In [ ]:
# Build the final export
final_output = {
    "rfp_name": "Sample RFP — Agent Engineering Challenge",
    "total_questions": len(RFP_QUESTIONS),
    "answers": all_answers if all_answers else [],
    "review": review if 'review' in dir() and review else "Review not completed",
    "metadata": {
        "model": "claude-sonnet-4-20250514",
        "knowledge_base_entries": len(KNOWLEDGE_BASE),
    }
}

print(json.dumps(final_output, indent=2))

---

## Part 8: ⭐ STRETCH — Eval Assertions

*This section is optional and unscored, but demonstrates rigor.*

Build assertions that validate your agent's output quality. Good evals test real failure modes, not just happy paths.

### What to Test

| Dimension | Example Assertion |
|-----------|-------------------|
| **Accuracy** | Q3 pricing answer contains "$18/seat/month" for 500 endpoints |
| **Source attribution** | Every answer has at least one source cited |
| **Consistency** | Q2 and Q5 don't contradict each other on certification dates |
| **Confidence calibration** | Questions with no KB match are flagged as low-confidence |
| **Edge cases** | What happens with an ambiguous or unanswerable question? |

In [ ]:
def run_evals(answers: list[dict]) -> dict:
    """Run quality assertions against agent output."""
    results = {"passed": 0, "failed": 0, "details": []}

    def record(question_id, assertion, passed, note=""):
        results["details"].append({
            "question": question_id,
            "assertion": assertion,
            "passed": passed,
            "note": note,
        })
        if passed:
            results["passed"] += 1
        else:
            results["failed"] += 1

    VALID_CONFIDENCE = {"high", "medium", "low"}
    REQUIRED_FIELDS = {"question_id", "category", "answer", "sources", "confidence", "flags"}

    for ans in answers:
        qid = ans.get("question_id", "?")

        # No API/parse error
        record(qid, "no_error", "error" not in ans and "parse_error" not in ans)

        # All required fields present and non-empty
        missing = [f for f in REQUIRED_FIELDS if not ans.get(f)]
        record(qid, "required_fields", not missing,
               note=f"missing: {missing}" if missing else "")

        # At least one source cited
        record(qid, "has_sources", len(ans.get("sources", [])) > 0)

        # confidence is a valid value
        conf = ans.get("confidence", "")
        record(qid, "valid_confidence", conf in VALID_CONFIDENCE,
               note=f"got: {conf!r}" if conf not in VALID_CONFIDENCE else "")

        # Answer is substantive (>50 chars)
        answer_text = ans.get("answer", "")
        record(qid, "answer_not_empty", len(answer_text) > 50,
               note=f"length={len(answer_text)}")

        # Q3 — pricing answer must include all three tier prices from the KB
        if qid == "Q3":
            for price in ("$18", "$15", "$11"):
                record(qid, f"pricing_contains_{price}", price in answer_text,
                       note="expected from pricing KB entry")

        # Q2 — compliance answer must mention key certifications
        if qid == "Q2":
            for cert in ("SOC 2", "ISO 27001", "FedRAMP", "PCI DSS", "HIPAA"):
                record(qid, f"compliance_mentions_{cert.replace(' ', '_')}",
                       cert.lower() in answer_text.lower(),
                       note="expected from compliance_certs KB entry")

    return results


if all_answers:
    eval_results = run_evals(all_answers)
    print(f"Eval Results: {eval_results['passed']} passed, {eval_results['failed']} failed\n")
    for detail in eval_results["details"]:
        status = "✓" if detail["passed"] else "❌"
        note = f"  ({detail['note']})" if detail.get("note") else ""
        print(f"  {status} {detail['question']}: {detail['assertion']}{note}")
else:
    print("⚠️ Complete Part 5 first")

---

## 🏁 Wrap-Up

There's no formal demo, but if you built something you're proud of — a clever architecture, a useful analysis UI, an unusual approach to consistency review, a robust eval harness — you're strongly encouraged to grab a few minutes and show the room. Past builds we've seen and loved:

- RFP deck builders that hand off to slide generators
- Eval harnesses with assertions richer than the baseline in Part 8
- Analysis UIs that visualize per-answer confidence and source coverage
- Search UIs that take any RFP file and run the agent end-to-end
- Subagent routers that dispatch by category

### Worth reflecting on (for yourself, or for anyone you show your build to):

1. How did you handle the tool use loop? Sequential or batch?
2. What does your review step actually catch?
3. What broke (or would break) when you pointed it at the surprise RFP?
4. If you had another hour, what would you build first?